# 02b — Chuẩn hóa bảng và đặc trưng vật lý

Giữ nguyên nguồn và nhãn. Tách bảng đầu vào khỏi nhãn, tính biến theo từng dòng. Chưa chia tập hoặc fit scaler trên AI4I; dùng công thức cố định không cần học thống kê từ toàn nguồn. Xem `docs/PREPROCESSING.md` để biết quyết định, kiểu dữ liệu và giới hạn.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "configs/ai4i-schema.json").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Mở notebook trong repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from maintenance.preprocessing import export_prepared
from maintenance.features import make_preprocessor
metadata = export_prepared(ROOT)
metadata

{'preprocessing_version': 'ai4i-preprocessing-v1',
 'schema_version': 'ai4i-v1',
 'dataset_version': 'sha256:dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e',
 'source_sha256': 'dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e',
 'rows_in': 10000,
 'rows_out': 10000,
 'rows_removed': 0,
 'labels_changed': 0,
 'scaler_fitted': False,
 'split_created': False,
 'label_flags': {'failure_without_mode': 9,
  'mode_without_failure': 18,
  'multiple_modes': 24},
 'feature_columns': ['product_type',
  'air_temperature_k',
  'process_temperature_k',
  'rotational_speed_rpm',
  'torque_nm',
  'tool_wear_min',
  'temperature_difference_k',
  'power_w',
  'wear_torque_min_nm'],
 'files': [{'path': 'data/interim/ai4i-readings.csv',
   'rows': 10000,
   'columns': ['dataset_version',
    'schema_version',
    'reading_id',
    'product_id',
    'product_type',
    'air_temperature_k',
    'process_temperature_k',
    'rotational_speed_rpm',
    'torque_nm',
    'tool_

## Pipeline dành cho bước mô hình

Hai cấu hình `base` và `physical` dùng cùng sáu đầu vào nguồn; `physical` bổ sung ba công thức. Type dùng one-hot theo miền L/M/H cố định từ schema. Bước chuẩn hóa số có thể bật cho phân lớp hoặc tắt khi dùng cây. Các đối tượng dưới đây CHƯA FIT.

Khi có split chung: chỉ gọi `.fit(X_train)` hoặc ghép vào Pipeline với estimator; validation/test chỉ `.transform(...)` hoặc `.predict(...)`. Không truyền bảng đã kèm ID/nhãn vào transformer. `select_inputs` là điểm tách sáu đầu vào từ bảng chuẩn hóa.

In [2]:
preprocessor_base = make_preprocessor("base")
preprocessor_physical = make_preprocessor("physical")
assert metadata["rows_in"] == metadata["rows_out"] == 10000
assert metadata["labels_changed"] == 0
assert metadata["scaler_fitted"] is False
print("Đã tạo bảng; hai pipeline chưa fit. Không tạo split hoặc mô hình ở bước này.")

Đã tạo bảng; hai pipeline chưa fit. Không tạo split hoặc mô hình ở bước này.
